# Step 1 - Prototype 2

We now have our first working prototype for step 1. In the second (and probably final) one, we mainly want to fix some smaller issues and decide on foundational formulations / implementations that will make extendability much easier later on. These are:
- Think of a good way to encode and implement train stations (probably already along with the corresponding constraints (essentially blocking movement until the leave time)).
- Adjust the optimization goal to minimize lateness (for now only at the destination station). Additionally, maybe make sure that the formulation can be extended in terms of travel and dwell time minimization goals later.

Adding a safety distance now might also be doable, however, this is in a sense then also connected to different train speeds (that might require different safety distances) and this is probably really then a point for step 2...

Small terminology recap (how I use it here):
- Track: What a train drives on
- Block: A section of tracks (1 or more) of fixed length (like if you would have a map on checkered paper)
- Segment: A sequence of (or also only one xD) block(s) with the same capacity

In [57]:
import gurobipy as gp
from gurobipy import GRB

import numpy as np
from enum import Enum

In [58]:
# Initialize the Model
model = gp.Model("Basic_MILP")

### Define the track system and timetable

Here we want to see, if it makes sense to create (separate?) definitions for train stations and block segments that can then hopefully be combined automatically or, if just adding segments of length one with a given 'station capacity' should work just fine (or even better xD).

Given the fact, that we probably only want to have leave and arrival times at stations and not on 'random' points on the tracks, a more native way of distinguishing (separate blocks and train stations) might be beneficial. This could then be handled via an additional boolean array `is_train_station_array` (or something like that xD).

**Another Idea:** `segment_lengths` is a list of lists (and single entries?). The same goes for segment capacities and per construction, there is always a station (with capacities specified in `station_capacities`) between a given connecting segment. This means, that `num_stations` is equal to `len(segment_lengths)`.

To avoid inhomogeneous arrays, we will use a list of dictionaries and a parser function that turns it into proper flat arrays for Gurobi... This way, we should be able to extend the model and add new features using similar helper / parsing functionalities, as well as later in the constraint generation. 

In [59]:
# Might use Enums or Dataclasses later but maybe also not xD
# class TrackType(Enum):
#     station = "station"
#     segment = "segment"

# Copy preset for faster building
# {"type": "station", "capacity": 0}
# {"type": "segment", "length": 0, "capacity": 0}

track_blueprint = [
    {"type": "station", "capacity": 3},
    {"type": "segment", "length": 5, "capacity": 2},
    {"type": "station", "capacity": 3},
    {"type": "segment", "length": 2, "capacity": 2},
    {"type": "station", "capacity": 3}
]

In [60]:
# Parsing the track blueprint and generating necessary data

block_capacities = []
is_station = []

for item in track_blueprint:
    if item["type"] == "station":
        block_capacities.append(item["capacity"])
        is_station.append(True)
    elif item["type"] == "segment":
        # Int conversion not necessary, but gets rid of IDE highlighting
        for _ in range(int(item["length"])):
            block_capacities.append(item["capacity"])
            is_station.append(False)
    else:
        print("typo? No other types yet...")

In [61]:
# Like with the boolean mask is_station, we will use a mask is_stop for our timetable as well.
# This will set us up nicely when having different train types (e.g. RB / RE and ICE) later on
# and also allows for a more flexible and complete rule set.

# New Idea: To allow for better extendability, we will try out a similar dictionary-based approach
# like we did with the track_blueprint. This will make it much easier to add essentially arbitrary
# amounts of additional information about each stop (e.g. dwell times, passengers, ...) and never
# have to worry about index-mismatches.

# Structure: [train_i_stop_information_dictionary{station_index_i: information{some_key: some_value}}]
# Note: Due to the implementation ('looking-back') leave_times_i has to be >= 1
# Suggestion: Probably rename 'leave' to 'departure' (throughout the code base)
train_schedules = [
    # Train 0
    {
        # Station x
        0: {"leave": 3},
        2: {"arrival": 16, "leave": 17},
    },
    # Train 1
    {
        # Station x
        1: {"leave": 5},

        # Think what dwell should / might mean in combination with a leave time ('overpower', something else?)
        # Might not be even implemented now but could be used later on...
        2: {"arrival": 15, "leave": 18, "dwell": 2},
    },
]

In [62]:
# Just a cell for dictionary access testing
print(sorted(train_schedules[1].keys())[0])

spawn_time_key = sorted(train_schedules[0].keys())[0]
spawn_time = train_schedules[0][spawn_time_key]["leave"]

print(spawn_time)

1
3


In [63]:
# Number of time steps over which we optimize
time_horizon = 25

# Deriving some additional variables
num_trains = len(train_schedules)
num_blocks = len(block_capacities)
num_stations = sum(is_station)

In [64]:
# Define Decision Variables

# To describe our train movement we need a way to know which train is where at what
# time (-> 3D time-position grid of boolean values)

x = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x")

### Constraints

#### General Constraints

As general constraints, we have the
- uniqueness constraint (no train can be on multiple tracks at once)
- spawning constraint (a train does not 'exist' until it leaves from the first station in its schedule). For future iterations we should also think about if we want to keep the spawning behavior or replace (or even remove?) it...
- capacity constraints (there can't exist more trains on a block (or station) than its capacity)

#### Movement Constraints

For now, we just have the 'at most one block per timestep' constraint. Will likely be changed in a later version though.

In [65]:
# Uniqueness constraint for trains to only exist once on a block (on only a single track at a given time)
# Additionally, we tackle the spawning (leave times), as before a train does not exist on any block
for i in range(num_trains):

    # Since dictionaries are unordered, use this to get spawn time (first leave time)
    spawn_time_key = sorted(train_schedules[i].keys())[0]
    spawn_time = train_schedules[i][spawn_time_key]["leave"]

    for k in range(time_horizon):

        # Sum over all blocks j to ensure that a train only exists once at a given time k
        active_tracks = gp.quicksum(x[i, j, k] for j in range(num_blocks))

        # If the train hasn't spawned (or left) yet, it can't exist on the track and we enforce this
        if k < spawn_time:
            model.addConstr(active_tracks == 0, name=f"NotSpawned_Train{i}_Time{k}")
        else:
            # If it already exists, we enforce that the train only exists on one track at any time k
            model.addConstr(active_tracks == 1, name=f"UniquePosition_Train{i}_Time{k}")

In [66]:
# Capacity constraints (the sum over all tracks j and all trains i <= capacities_i)
for k in range(time_horizon):

    # Here we want to sum over all i AND j and add the constraint that this should be smaller than
    # Is this formulation correct and can this be made more compact (yes: x.sum('*', j, k))?
    for j in range(num_blocks):
        occupied_tracks = gp.quicksum(x[i, j, k] for i in range(num_trains))

        # Add constraint to enforce capacity maximum across block j
        model.addConstr(occupied_tracks <= block_capacities[j], name=f"Capacity_Block{j}_Time{k}")

In [67]:
# Movement Constraints

# A train can (after spawning) either wait or move
for i in range(num_trains):

    # (Copied, consider helper function or dedicated list as a global variable)
    # Since dictionaries are unordered, use this to get spawn time (first leave time)
    spawn_time_key = sorted(train_schedules[i].keys())[0]
    spawn_time = train_schedules[i][spawn_time_key]["leave"]

    # At every time point, we need to differentiate between different scenarios
    for k in range(time_horizon):

        # If the train hasn't left yet, it's not yet on the grid and the spawn constraint takes care of this
        if k < spawn_time:
            pass
        # Here the train spawns, and we fix a position (the block that corresponds to the first station on
        # the trains schedule)
        elif k == spawn_time:

            # Conveniently, the spawn_time_key is the spawn station index
            station_block_indices = [block for block, station_idx in enumerate(is_station) if station_idx]
            block_index = station_block_indices[spawn_time_key]

            model.addConstr(x[i, block_index, k] == 1, name=f"Spawn_Train{i}_Time{k}_on_Block0")
        else:

            # Handle out back-looking approach
            for j in range(num_blocks):

                # If the train is on block one, it must have already been there (j is always ≠ -1)
                if j == 0:
                    model.addConstr(x[i, 0, k] <= x[i, 0, k - 1], name=f"Move_Train{i}_Time{k}_Block0")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x[i, j, k] <= x[i, j, k - 1] + x[i, j - 1, k - 1], name=f"Move_Train{i}_Time{k}_Block{j}")

In [68]:
# Now, we want to add the constraints for intermediate stations (arrival and departure times and no spawning)
# Note: With a few 'helper functions / lines' we might be able to reduce verbosity in this code cell...

# First, only for departure times (and no dwell times)

# Block indices of all stations of our straight track
# This is a duplicate line (consider making global)
station_block_indices = [block for block, station_idx in enumerate(is_station) if station_idx]
print(is_station)
print(station_block_indices)

# For every train
for i in range(num_trains):

    # Get the spawn station ID so we can skip it (this is already handled by the spawning constraint)
    spawn_station_id = sorted(train_schedules[i].keys())[0]

    # Iterate directly through this train's schedule
    for station_id, schedule in train_schedules[i].items():

        # Skip the spawn station (already handled)
        if station_id == spawn_station_id:
            continue

        # Check if this station has a strict leave time (might always be the case, but we'll see)
        if "leave" in schedule:
            leave_time = schedule["leave"]

            # print(station_block_indices, station_id, spawn_station_id)

            station_block = station_block_indices[station_id]
            next_block = station_block + 1

            # Ensure we don't look past the end of the tracks (should only be an issue with the last station)
            if next_block < num_blocks:

                # Build the wall: The train cannot enter the next block before the leave time
                for k in range(leave_time):
                    model.addConstr(
                        x[i, next_block, k] == 0,
                        name=f"DepartureWall_Train{i}_Block{next_block}_Time{k}"
                    )
            else:
                # Here we might have some code later when having more complex schedules and a train e.g.,
                # 'turns around' and continues its journey back to the initial station. However, this
                # might then also be handled differently again...
                pass

[True, False, False, False, False, False, True, False, False, True]
[0, 6, 9]


In [69]:
# Set the Model Objective (still needs to be revamped for prototype 2)

# Here we basically only want to minimize the final delay (actual_arrival_time - timetable_arrival_time)

# Now though, just for testing purposes, we will choose a simpler to implement but later 'stupid' goal:
# Maximize the time spent at the destination.

last_block = num_blocks - 1

objective = gp.quicksum(x[i, last_block, k] for i in range(num_trains) for k in range(time_horizon))
model.setObjective(objective, GRB.MAXIMIZE)

In [70]:
# Run the optimization
model.optimize()

# Code by Gemini

# Print the results if an optimal solution is found
if model.Status == GRB.OPTIMAL:
    print("\n" + "="*30)
    print("🚂 OPTIMAL SCHEDULE FOUND 🚂")
    print("="*30)

    for i in range(num_trains):
        print(f"\n--- Train {i} ---")
        for k in range(time_horizon):
            for j in range(num_blocks):
                # .X extracts the solved value from the Gurobi variable.
                # We check > 0.5 to account for minor floating-point tolerances in the solver.
                if x[i, j, k].X > 0.5:
                    print(f"Time {k:2d} -> Block {j}")
else:
    print("\n❌ No optimal solution found. The model might be Infeasible.")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M3 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 702 rows, 500 columns and 2162 nonzeros
Model fingerprint: 0x57f98dcf
Variable types: 0 continuous, 500 integer (500 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]
Presolve removed 520 rows and 333 columns
Presolve time: 0.00s
Presolved: 182 rows, 167 columns, 636 nonzeros
Variable types: 0 continuous, 167 integer (167 binary)
Found heuristic solution: objective 17.0000000
Found heuristic solution: objective 30.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.01 work units)
Thread count was 12 (of 12 available processors)

Solution count 2: 30 17 

Optimal solution found (tolerance 1.00e-04)
Best objective 3.000000000000e+01, best bound 3.00